In [1]:
from typing import TypedDict
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
load_dotenv(override=True)

/home/ubuntu/My_learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatAnthropic(
    api_key=os.environ.get("KEY") or os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0"),
    base_url=os.environ.get("ENDPOINT"),
    temperature=0.3,
    max_tokens=300,
)

In [3]:

class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str


def review_node(state: ReviewState) -> dict:
    prompt = (
        "Write a short customer review (2-3 sentences) for this product:\n"
        f"{state['product']}\n\n"
        "Keep it realistic and concise."
    )
    response = llm.invoke(prompt)
    return {"review": response.content.strip()}


def sentiment_node(state: ReviewState) -> dict:
    prompt = (
        "Classify the sentiment of this review as exactly one word: "
        "Positive, Negative, or Neutral.\n\n"
        f"Review:\n{state['review']}\n\n"
        "Output only one of: Positive | Negative | Neutral"
    )
    response = llm.invoke(prompt).content.strip()

    # Safety normalization
    normalized = response.lower()
    if "positive" in normalized:
        sentiment = "Positive"
    elif "negative" in normalized:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return {"sentiment": sentiment}


def reply_node(state: ReviewState) -> dict:
    prompt = (
        "You are a brand support representative.\n"
        "Write a one-line response to the customer based on the review sentiment.\n"
        f"Sentiment: {state['sentiment']}\n"
        f"Review: {state['review']}\n\n"
        "Rules: One sentence only, polite and professional."
    )
    response = llm.invoke(prompt)
    return {"reply": response.content.strip()}



builder = StateGraph(ReviewState)

builder.add_node("review_node", review_node)
builder.add_node("sentiment_node", sentiment_node)
builder.add_node("reply_node", reply_node)

builder.add_edge(START, "review_node")
builder.add_edge("review_node", "sentiment_node")
builder.add_edge("sentiment_node", "reply_node")
builder.add_edge("reply_node", END)

graph = builder.compile()


result = graph.invoke(
    {
        "product": "wireless noise-cancelling headphones",
        "review": "",
        "sentiment": "",
        "reply": "",
    }
)

print("Product   :", result["product"])
print("Review    :", result["review"])
print("Sentiment :", result["sentiment"])
print("Reply     :", result["reply"])

Product   : wireless noise-cancelling headphones
Review    : # Customer Review: Wireless Noise-Cancelling Headphones

These headphones deliver excellent noise cancellation and the battery lasts all week with regular use, making them perfect for commuting and travel. The sound quality is crisp and balanced, though they're a bit snug after wearing them for several hours. Overall, great value for the price and I'd definitely recommend them to anyone looking for reliable wireless headphones.
Sentiment : Positive
Reply     : Thank you so much for the wonderful review—we're thrilled that our Wireless Noise-Cancelling Headphones are enhancing your commute and travel experiences, and we appreciate your recommendation!
